In [ ]:
import pandas as pd
import numpy as np

In [ ]:
creditCardData = pd.read_csv('../input/creditcard.csv')

In [ ]:
from sklearn.preprocessing import StandardScaler
creditCardData['normAmount'] = StandardScaler().fit_transform(creditCardData['Amount'].values.reshape(-1, 1))
creditCardData = creditCardData.drop(['Time','Amount'],axis=1)
creditCardData.head()

In [ ]:
X = creditCardData.iloc[:, creditCardData.columns != 'Class']
y = creditCardData.iloc[:, creditCardData.columns == 'Class']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3)

In [ ]:
print(np.shape(X_train))
print(np.shape(y_train))
print(np.shape(X_test))
print(np.shape(y_test))

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.utils import to_categorical

In [ ]:
NN = Sequential()
NN.add(Dense(units=16,kernel_initializer='uniform',activation='relu',input_dim=29))
NN.add(Dropout(rate=0.1))
NN.add(Dense(units=16,kernel_initializer='uniform',activation='relu',input_dim=29))
NN.add(Dropout(rate=0.1))
NN.add(Dense(units=1,kernel_initializer='uniform',activation='sigmoid'))
NN.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [ ]:
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [ ]:
history = NN.fit(X_train, y_train, batch_size=20, epochs=100, callbacks=[ReduceLROnPlateau(patience=3, verbose=1, min_lr=1e-6), EarlyStopping(patience=5, verbose=1)])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
num_epochs = len(history.history["loss"])
fig, axarr = plt.subplots(1, 2, figsize=(24, 8))
axarr[0].set_xlabel("Number of Epochs")
axarr[0].set_ylabel("Loss")
sns.lineplot(x=range(1, num_epochs+1), y=history.history["loss"], label="Train", ax=axarr[0])
axarr[1].set_xlabel("Number of Epochs")
axarr[1].set_ylabel("Accuracy")
sns.lineplot(x=range(1, num_epochs+1), y=history.history["acc"], label="Train", ax=axarr[1])

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix, classification_report, matthews_corrcoef, cohen_kappa_score, accuracy_score, average_precision_score, roc_auc_score,auc,precision_recall_curve,roc_curve

In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
def classification_eval(estimator, X_test, y_test):
    y_pred = estimator.predict(X_test)
    y_score = np.argmax(y_pred, axis=1)
    dec = np.int64(np.ceil(np.log10(len(y_test))))
    
    def plot_roc():
        fpr, tpr, t = roc_curve(y_test, y_pred)
        plt.plot(fpr, tpr, label = 'ROC curve', linewidth = 2)
        plt.plot([0,1],[0,1], 'k--', linewidth = 2)
        plt.xlim([0.0,0.001])
        plt.ylim([0.0,1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve (AUC=%0.2f)'% auc(fpr,tpr))
        plt.show();
    
    def plot_precision_recall():
        precision, recall, thresholds = precision_recall_curve(y_test, y_pred)
        plt.step(recall, precision, color = 'b', alpha = 0.2,where = 'post')
        plt.fill_between(recall, precision, step ='post', alpha = 0.2,color = 'b')
        plt.plot(recall, precision, linewidth=2)
        plt.xlim([0.0,1])
        plt.ylim([0.0,1.05])
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision Recall Curve')
        plt.show();

    print('CONFUSION MATRIX')
    cfm=confusion_matrix(y_test, y_score)
    plot_confusion_matrix(cfm, classes = range(2)) 
    plt.show()
    print('CLASSIFICATION REPORT')
    print(classification_report(y_test, y_score, digits=dec))
    print('SCALAR METRICS')
    format_str = '%%13s = %%.%if' % dec
    print(format_str % ('Accuracy', accuracy_score(y_test, y_score)))
    plot_roc()
    plot_precision_recall()

In [ ]:
classification_eval(NN, X_test, y_test)